In [1]:
#(TASK 2 CREDIT CARD FRAUD DATECTION)

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [3]:
#loading dataset
train_df = pd.read_csv("fraudTrain.csv")
test_df = pd.read_csv("fraudTest.csv")

print("Train Shape :", train_df.shape)
print("Test Shape :", test_df.shape)

train_df.head()

Train Shape : (1296675, 23)
Test Shape : (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [4]:
#creating useful feature
# Convert transaction time

train_df["trans_date_trans_time"] = pd.to_datetime(
    train_df["trans_date_trans_time"]
)

test_df["trans_date_trans_time"] = pd.to_datetime(
    test_df["trans_date_trans_time"]
)

# Extract hour

train_df["hour"] = train_df["trans_date_trans_time"].dt.hour
test_df["hour"] = test_df["trans_date_trans_time"].dt.hour

# Extract day

train_df["day"] = train_df["trans_date_trans_time"].dt.day
test_df["day"] = test_df["trans_date_trans_time"].dt.day

# Extract month

train_df["month"] = train_df["trans_date_trans_time"].dt.month
test_df["month"] = test_df["trans_date_trans_time"].dt.month

print("Feature Engineering Completed")

Feature Engineering Completed


In [5]:
#creating column
selected_columns = [

    "merchant",
    "category",
    "amt",
    "gender",
    "state",
    "city_pop",
    "hour",
    "day",
    "month"

]

In [6]:
#encode categorical data
encoder_dict = {}

for col in ["merchant","category","gender","state"]:

    encoder = LabelEncoder()

    combined = pd.concat([
        train_df[col],
        test_df[col]
    ])

    encoder.fit(combined)

    train_df[col] = encoder.transform(train_df[col])

    test_df[col] = encoder.transform(test_df[col])

    encoder_dict[col] = encoder

print("Encoding Completed")

Encoding Completed


In [7]:
#prepare data
X_train = train_df[selected_columns]

y_train = train_df["is_fraud"]

X_test = test_df[selected_columns]

y_test = test_df["is_fraud"]

print(X_train.shape)
print(X_test.shape)

(1296675, 9)
(555719, 9)


In [8]:
# Fix missing values in target column

y_train = y_train.fillna(0)
y_test = y_test.fillna(0)

# Fix missing values in feature columns

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Train Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear",
    class_weight="balanced"
)

lr_model.fit(
    X_train,
    y_train
)

lr_predictions = lr_model.predict(
    X_test
)

print("Logistic Regression Results")

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        lr_predictions
    )
)

print(
    classification_report(
        y_test,
        lr_predictions
    )
)

Logistic Regression Results
Accuracy: 0.9625296237846825
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    553574
           1       0.07      0.74      0.13      2145

    accuracy                           0.96    555719
   macro avg       0.54      0.85      0.56    555719
weighted avg       1.00      0.96      0.98    555719



In [9]:
#random forest
rf_model = RandomForestClassifier(

    n_estimators=120,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

rf_model.fit(
    X_train,
    y_train
)

rf_predictions = rf_model.predict(
    X_test
)

print("Random Forest Results")

print(
    "Accuracy :",
    accuracy_score(
        y_test,
        rf_predictions
    )
)

print(
    "Precision :",
    precision_score(
        y_test,
        rf_predictions
    )
)

print(
    "Recall :",
    recall_score(
        y_test,
        rf_predictions
    )
)

print(
    "F1 Score :",
    f1_score(
        y_test,
        rf_predictions
    )
)

Random Forest Results
Accuracy : 0.9983930727579946
Precision : 0.9007682458386683
Recall : 0.6559440559440559
F1 Score : 0.7591043970865929


In [12]:
#compare models
lr_acc = accuracy_score(
    y_test,
    lr_predictions
)

rf_acc = accuracy_score(
    y_test,
    rf_predictions
)

print("Logistic Regression :", lr_acc)

print("Random Forest :", rf_acc)

if rf_acc > lr_acc:
    print("Best Model = Random Forest")
else:
    print("Best Model = Logistic Regression")

Logistic Regression : 0.9625296237846825
Random Forest : 0.9983930727579946
Best Model = Random Forest


In [13]:
#saving fraud predictions
output = test_df.copy()

output["Predicted_Fraud"] = rf_predictions

output[
    [
        "amt",
        "category",
        "Predicted_Fraud"
    ]
].to_csv(
    "fraud_predictions.csv",
    index=False
)

print("fraud_predictions.csv saved")

fraud_predictions.csv saved
